# Slideflow MSI Results Explorer

This notebook is an open-source, inbuilt results viewer for the TCGA-CRC Slideflow MSI workflow.

It is designed for demo and detailed result review using:

- `pandas`
- `plotly`
- `matplotlib`
- optional `panel` if you want to extend it into a dashboard later


## VM Setup

Run this on the VM with the `pathology310` kernel.

Expected result files live under:

`project_1_slideflow_msi_tcga_crc/slideflow_project/results/`


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path.cwd().resolve().parents[1]
RESULTS_DIR = PROJECT_DIR / 'slideflow_project' / 'results'

SUMMARY_PATH = RESULTS_DIR / 'cv_summary.json'
TABLE_PATH = RESULTS_DIR / 'cv_auroc_table.csv'
ROC_PATH = RESULTS_DIR / 'roc_curves.csv'

print('PROJECT_DIR:', PROJECT_DIR)
print('RESULTS_DIR:', RESULTS_DIR)
print('SUMMARY_PATH exists:', SUMMARY_PATH.exists())
print('TABLE_PATH exists:', TABLE_PATH.exists())
print('ROC_PATH exists:', ROC_PATH.exists())


In [ ]:
summary = json.loads(SUMMARY_PATH.read_text()) if SUMMARY_PATH.exists() else {}
table_df = pd.read_csv(TABLE_PATH) if TABLE_PATH.exists() else pd.DataFrame()
roc_df = pd.read_csv(ROC_PATH) if ROC_PATH.exists() else pd.DataFrame()

display(pd.DataFrame([summary]) if summary else pd.DataFrame())
display(table_df)
display(roc_df.head() if not roc_df.empty else roc_df)


In [ ]:
try:
    import plotly.express as px
    import plotly.graph_objects as go

    if not table_df.empty:
        fig_bar = px.bar(
            table_df,
            x='fold',
            y='auroc',
            text='auroc',
            color='auroc',
            title='Fold-level AUROC Overview',
            color_continuous_scale='Blues'
        )
        fig_bar.update_traces(texttemplate='%{text:.3f}', textposition='outside')
        fig_bar.update_layout(yaxis_range=[0.0, 1.0])
        fig_bar.show()

        fig_n = px.scatter(
            table_df,
            x='fold',
            y='n',
            size='n',
            color='auroc',
            hover_data=['score_column', 'file'],
            title='Validation Sample Count And AUROC By Fold',
            color_continuous_scale='Viridis'
        )
        fig_n.show()

    if not roc_df.empty:
        fig_roc = px.line(
            roc_df,
            x='fpr',
            y='tpr',
            color='fold',
            title='ROC Curves Across Folds'
        )
        fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='chance', line=dict(dash='dash')))
        fig_roc.show()
except Exception as exc:
    print('Interactive Plotly view unavailable:', exc)


In [ ]:
try:
    import matplotlib.pyplot as plt

    if not table_df.empty:
        ax = table_df.plot(kind='bar', x='fold', y='auroc', legend=False, figsize=(8, 4), title='Fold-level AUROC')
        ax.set_ylim(0, 1)
        plt.show()

    if not roc_df.empty:
        fig, ax = plt.subplots(figsize=(7, 5))
        for fold, fold_df in roc_df.groupby('fold'):
            ax.plot(fold_df['fpr'], fold_df['tpr'], label=f'Fold {fold}')
        ax.plot([0, 1], [0, 1], linestyle='--', label='chance')
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title('ROC Curves Across Folds')
        ax.legend()
        plt.show()
except Exception as exc:
    print('Matplotlib fallback unavailable:', exc)


## What To Review In Detail

- fold-to-fold AUROC stability
- which prediction file each fold came from
- whether one fold has unusually low sample count or weaker performance
- whether ROC shape is consistent across folds
- whether the downstream heatmaps match tissue intuition from slide review
